# BlinkIT Grocery Sales Analysis

**Business question:** What product and outlet characteristics distinguish higher recorded sales, and where should management investigate improvement opportunities?

This notebook follows the Sapphire IQ student brief. It examines data reliability, product/category performance, outlet and location comparisons, and practical recommendations. `Sales` means the value recorded in the source workbook. Results are descriptive; no dates, costs, margins, or transaction IDs are supplied.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.analysis import load_data, clean_data
DATA_PATH = ROOT / "data" / "blinkit_grocery_data.xlsx"
raw = load_data(DATA_PATH)
df = clean_data(raw)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

## 1. Understand the available records

In [ ]:
display(df.head())
df.info()

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
})
display(quality)
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
display(df.describe(include="all").T)

### Data preparation notes

The source has 1,463 missing `Item Weight` values. They remain missing rather than being filled with a fabricated value; weight analyses use available observations only. Fat content labels are harmonized (`LF` and `low fat` → `Low Fat`; `reg` → `Regular`). Exact duplicate rows are checked, but the file has no transaction identifier with which to identify repeated business events. Sales and rating bounds are inspected below; statistical outliers are flagged for investigation rather than automatically removed.

In [ ]:
display(raw["Item Fat Content"].value_counts(dropna=False).rename("source_count").to_frame())
display(df["Item Fat Content"].value_counts(dropna=False).rename("standardized_count").to_frame())
for col in ["Sales", "Item Visibility", "Item Weight"]:
    print(f"{col}: min={df[col].min():.3f}, median={df[col].median():.3f}, max={df[col].max():.3f}")
    q1, q3 = df[col].quantile([.25,.75]); iqr=q3-q1
    print(f"  IQR review flags: {((df[col] < q1-1.5*iqr) | (df[col] > q3+1.5*iqr)).sum():,}")

## 2. Overall performance

In [ ]:
summary = pd.Series({
    "Records": len(df),
    "Total recorded sales": df["Sales"].sum(),
    "Mean sales per record": df["Sales"].mean(),
    "Median sales per record": df["Sales"].median(),
    "Mean customer rating": df["Rating"].mean(),
    "Missing item weight records": df["Item Weight"].isna().sum(),
})
display(summary.to_frame("value").round(2))

## 3. Product and category performance

In [ ]:
category = df.groupby("Item Type")["Sales"].agg(total_sales="sum", mean_sales="mean", records="size").sort_values("total_sales", ascending=False)
display(category.round(2))
ax=category["total_sales"].sort_values().plot(kind="barh",figsize=(10,7),color=sns.color_palette("viridis",len(category)))
ax.set(title="Recorded sales by item type",xlabel="Sales",ylabel="Item type"); plt.tight_layout()

In [ ]:
fat = df.groupby("Item Fat Content")["Sales"].agg(total_sales="sum", mean_sales="mean", records="size")
display(fat.round(2))
fig, ax = plt.subplots(figsize=(8,4)); sns.boxplot(data=df,x="Item Fat Content",y="Sales",showfliers=False,ax=ax)
ax.set(title="Sales distribution by standardized fat content",xlabel="Fat content",ylabel="Sales per record"); plt.tight_layout()

## 4. Outlet and location performance

In [ ]:
outlet_type = df.groupby("Outlet Type")["Sales"].agg(total_sales="sum", mean_sales="mean", records="size")
outlet_type["sales_share_pct"] = outlet_type.total_sales / df.Sales.sum() * 100
display(outlet_type.sort_values("total_sales",ascending=False).round(2))
fig, ax=plt.subplots(figsize=(9,5)); sns.barplot(data=df,x="Outlet Type",y="Sales",estimator=sum,errorbar=None,ax=ax)
ax.tick_params(axis="x",rotation=20); ax.set(title="Recorded sales by outlet type",ylabel="Total sales",xlabel="Outlet type"); plt.tight_layout()

In [ ]:
location = df.groupby("Outlet Location Type")["Sales"].agg(total_sales="sum", mean_sales="mean", records="size")
size = df.groupby("Outlet Size")["Sales"].agg(total_sales="sum", mean_sales="mean", records="size")
outlets = df.groupby("Outlet Identifier")["Sales"].agg(total_sales="sum", mean_sales="mean", records="size").sort_values("total_sales",ascending=False)
display(location.round(2)); display(size.round(2)); display(outlets.round(2))

## 5. Product characteristics and recorded sales

In [ ]:
corr = df[["Sales","Item Visibility","Item Weight","Rating"]].corr(method="pearson")
display(corr.round(3))
fig, axes=plt.subplots(1,2,figsize=(12,4))
sns.regplot(data=df,x="Item Visibility",y="Sales",scatter_kws={"alpha":.15,"s":12},line_kws={"color":"crimson"},ax=axes[0]); axes[0].set_title("Visibility and sales (descriptive)")
sns.regplot(data=df,x="Rating",y="Sales",x_jitter=.08,scatter_kws={"alpha":.12,"s":12},line_kws={"color":"crimson"},ax=axes[1]); axes[1].set_title("Rating and sales (descriptive)")
plt.tight_layout()

## 6. Management takeaways and recommendations

In [ ]:
top_categories = category.head(3).index.tolist()
best_type = outlet_type.total_sales.idxmax()
best_location = location.total_sales.idxmax()
print(f"1. Protect availability in leading item types: {', '.join(top_categories)} lead the dataset by total recorded sales.")
print(f"2. Review {best_type} for scalable practices, while comparing per-record sales and outlet context; its total partly reflects {int(outlet_type.loc[best_type,'records']):,} records.")
print(f"3. Investigate {best_location} outlets using local assortment, traffic, and operating context before reallocating resources; its total is also affected by record counts.")
print("4. Improve item-weight completeness (17.2% missing) and add transaction dates, costs/margins, and transaction IDs to support trend, profitability, and event-level analysis.")
print("5. Visibility's near-zero linear correlation with sales here is not evidence that placement never matters; test with controlled experiments or richer covariates.")

## 7. Conclusion and limitations

The file describes sales records across products and outlets, with almost identical mean record sales by outlet location and type. Higher total sales for particular categories, outlet types, sizes, or tiers must be interpreted alongside record counts. Data gaps and the absence of time, cost, and transaction fields limit trend, margin, and causal conclusions. Recommendations should be tested with operational context and richer data before major investment decisions.